# OLS: The Normal Equation

[← Back to wiki](https://ml-viz.vercel.app/wiki/ols-normal-equation)

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## From-scratch OLS

In [ ]:
def ols(X, y):
    """w* = (X^T X)^{-1} X^T y  (solved via np.linalg.solve for numerical stability)"""
    return np.linalg.solve(X.T @ X, X.T @ y)

def add_bias(X):
    """Prepend a column of ones for the bias term w0."""
    return np.column_stack([np.ones(len(X)), X])

def r_squared(y, y_hat):
    ss_res = np.sum((y - y_hat)**2)
    ss_tot = np.sum((y - y.mean())**2)
    return 1 - ss_res / ss_tot

# Wiki worked example: 3 points
X_raw = np.array([1., 2., 3.])
y     = np.array([2., 3., 5.])
X = add_bias(X_raw)

w = ols(X, y)
y_hat = X @ w
print(f"w = {w.round(3)}")          # [0.333, 1.5]
print(f"Residuals: {(y-y_hat).round(3)}")  # [ 0.167, -0.333, 0.167]
print(f"R² = {r_squared(y,y_hat):.3f}")    # 0.964
print(f"Prediction at x=4: {np.array([1,4]) @ w:.3f}")  # 6.333

## OLS vs. lstsq vs. Ridge on synthetic data

In [ ]:
rng = np.random.default_rng(42)
n, d = 50, 3
X_syn = rng.standard_normal((n, d))
w_true = np.array([1.0, -2.0, 0.5])
y_syn = X_syn @ w_true + 0.3*rng.standard_normal(n)

X_syn_b = add_bias(X_syn)

# OLS
w_ols = ols(X_syn_b, y_syn)

# lstsq (handles near-singular X^TX gracefully)
w_ls, *_ = np.linalg.lstsq(X_syn_b, y_syn, rcond=None)

print(f"True w:   {w_true}")
print(f"OLS  w:   {w_ols[1:].round(3)}")   # skip bias
print(f"lstsq w:  {w_ls[1:].round(3)}")

## Residual plot

In [ ]:
y_hat_syn = X_syn_b @ w_ols
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_syn, y_hat_syn, alpha=0.6, color='#6366f1')
mn, mx = min(y_syn.min(),y_hat_syn.min()), max(y_syn.max(),y_hat_syn.max())
axes[0].plot([mn,mx],[mn,mx],'--',color='#f59e0b'); axes[0].set_xlabel('True y'); axes[0].set_ylabel('Predicted y'); axes[0].set_title('Predicted vs True')
axes[1].scatter(y_hat_syn, y_syn-y_hat_syn, alpha=0.6, color='#6366f1')
axes[1].axhline(0, color='#f59e0b', linestyle='--'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual'); axes[1].set_title('Residual plot')
plt.tight_layout(); plt.show(); print(f"R² = {r_squared(y_syn, y_hat_syn):.3f}")

## ✏️ Your turn

**Task:** Generate a dataset where $X^TX$ is nearly singular (two highly correlated features). Observe what happens to the OLS weights, then compare with Ridge regression.

```python
X_corr = np.column_stack([X_syn[:,0], X_syn[:,0] + 0.01*rng.standard_normal(n)])
```

In [ ]:
# TODO(you): fit OLS on X_corr, observe large/unstable weights
# Then fit Ridge: w_ridge = np.linalg.solve(X.T@X + lam*np.eye(d), X.T@y)

In [ ]:
# assert: ridge weights should have smaller L2 norm than OLS weights
# assert np.linalg.norm(w_ridge) < np.linalg.norm(w_ols_corr)

<details><summary>Solution</summary>

```python
lam = 1.0
X_c = add_bias(np.column_stack([X_syn[:,0], X_syn[:,0]+0.01*rng.standard_normal(n)]))
w_ols_c = ols(X_c, y_syn)
w_ridge = np.linalg.solve(X_c.T@X_c + lam*np.eye(X_c.shape[1]), X_c.T@y_syn)
print(f'OLS   |w|={np.linalg.norm(w_ols_c):.2f}')
print(f'Ridge |w|={np.linalg.norm(w_ridge):.2f}')
# Ridge is much smaller — regularization tames the ill-conditioning
```
</details>